
<h1 style="text-align: center;">CIÊNCIA DE DADOS</h1>
<h1 style="text-align: center;">Roteiro de Atividade Prática</h1>
<br>
<br>

Nome: ______________________________________________________________________________________      

Turma: ______________


**Componente:** Inteligência Artificial 
<br>
**Unidade Curricular:** Redes Neurais
<br>
**Tema da Semana:** Redes Neurais Recorrentes (RNNs)
<br>


# Aula 3: Exercícios de Fixação

## Tarefa
- Execute o código abaixo.

- Se as bibliotecas não tiverem instaladas, instale-as.

- Acompanhe as instruções.

- Preencha o código onde está #TO DO

- Responda as perguntas e conclua a atividade


### Código

### Instalação das dependência do tensorflow

In [ ]:
#!pip install --upgrade pip setuptools wheel

In [ ]:
#!pip install tensorflow

In [ ]:
#!conda install anaconda::tensorflow

In [ ]:
#!pip install numpy pandas matplotlib

### Imports de dependências

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
from numpy.lib.stride_tricks import sliding_window_view
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, SimpleRNN
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.initializers import GlorotUniform

### Configurações

In [ ]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

### Leitura dos dados

In [ ]:
df = pd.read_csv('heart-rate-time-series.csv', names=['Data'])

In [ ]:
# Número total de pontos no tempo (quantidade total de dados na série temporal)
N = df.shape[0] 

# Ponto no tempo para separar os dados de treino e teste (usando 30% dos dados para teste)
Tp = int( N - (N * 0.30)) 

### Apresentação dos dados

In [ ]:
plt.figure(figsize=(15,4))
plt.plot(df,c='blue',linewidth=2)
plt.grid(True)
plt.show()

### Separaçao em dos dados em Treino e Teste

In [ ]:
values=df.values
train,test = values[0:Tp,:], values[Tp:N,:]

### Apresentação dos dados de Treino e Teste

In [ ]:
index = df.index.values
plt.figure(figsize=(15,4))
plt.plot(index[0:Tp],train,c='blue', linewidth=2)
plt.plot(index[Tp:N],test,c='orange',alpha=0.7, linewidth=2)
plt.legend(['Train','Test'])
plt.axvline(df.index[Tp], c="r")
plt.grid(True)
plt.show()

### Ajuste dos dados para matrix com 4 ponto no tempo

In [ ]:
steps = 4

test = np.append(test, np.repeat(test[-1,], steps))
train = np.append(train, np.repeat(train[-1,], steps))

In [ ]:
def generate_time_steps(data, step):
    X = sliding_window_view(data, window_shape=step)[:-1]
    Y = data[step:]
    return X, Y

In [ ]:
# Converte os dados de treino e teste
X_train, y_train = generate_time_steps(train, steps)
X_test, y_test = generate_time_steps(test, steps)

In [ ]:
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

### Criação o modelo

In [ ]:
model = Sequential()

# Unidades recorrentes (SimpleRNN): A camada SimpleRNN é responsável por processar cada elemento da sequência de dados. 
# Cada unidade recebe uma entrada e o estado oculto da etapa anterior.
# O estado oculto é mantido entre as unidades recorrentes e carrega informações anteriores da sequência, 
# permitindo à rede aprender dependências temporais.

 #TO DO
units = 5
#units = 15

model.add(SimpleRNN(units=units, input_shape=(1, steps), activation="relu",
                    kernel_initializer=GlorotUniform(seed=42)))
model.add(Dropout(0.1, seed=SEED))
model.add(Dense(32, activation="relu"))
model.add(Dense(1))

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error',  metrics=['mse'])

### Treinamento do modelo

In [ ]:
batch_size=32
num_epochs = 60

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
# O treinamento da RNN ocorre com BPTT Backpropagation Through Time (BPTT), 
# ajustando os pesos ao longo de várias etapas temporais para minimizar o erro total.
history = model.fit(X_train, y_train, epochs=num_epochs, batch_size=128, validation_data=(X_test, y_test), callbacks=[early_stopping], verbose=0)

### Avaliação o modelo

In [ ]:
def plot_training_history(history, early_stopping, title):
    best_epoch = early_stopping.stopped_epoch - early_stopping.patience + 1
    
    epochs = range(1, len(history.history['loss']) + 1)
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history.history['loss'], label='Perda - Treinamento')
    plt.plot(epochs, history.history['val_loss'], label='Perda - Validação')
    plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Melhor Época (Época {best_epoch})')
    plt.title(f'Curva de Perda - {title}')
    plt.xlabel('Épocas')
    plt.ylabel('Perda')
    plt.legend()
    plt.xticks(epochs)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, np.sqrt(history.history['mse']), label='RMSE - Treinamento')
    plt.plot(epochs, np.sqrt(history.history['val_mse']), label='RMSE - Validação')
    plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Melhor Época (Época {best_epoch})')
    plt.title(f'Curva de RMSE - {title}')
    plt.xlabel('Épocas')
    plt.ylabel('Acurácia')
    plt.legend()
    plt.xticks(epochs)
    
    # Mostrar o gráfico
    plt.show()

    return best_epoch

def print_best_epoch_metrics(history, best_epoch):
    best_val_loss = history.history['val_loss'][best_epoch-1]
    best_val_rmse = np.sqrt(history.history['val_mse'])[best_epoch-1]

    print(f"\nMelhor Época: {best_epoch}")
    print(f"val_loss: {best_val_loss:.4f}")
    print(f"val_rmse: {best_val_rmse:.4f}")

In [ ]:
best_epoch = plot_training_history(history, early_stopping, title="")
print_best_epoch_metrics(history, best_epoch)

In [ ]:
train_predict = model.predict(X_train)
test_predict = model.predict(X_test)
predicted = np.concatenate((train_predict, test_predict), axis=0)

index = df.index.values
plt.figure(figsize=(15, 4))
plt.title("Valores Reais e Predições", fontsize=18)

plt.plot(index, df, c='blue', linewidth=2)
plt.plot(index, predicted, c='orange', alpha=0.75, linewidth=2)
plt.legend(['Valor Real', 'Predição'], fontsize=15)
plt.axvline(df.index[Tp], c="r")
plt.grid(True)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.show()

### Perguntas e Conclusão

### Crie uma cópia do notebook e aumente a quantidade de units para 15.
##### As métricas val_loss e val_rmse ficaram melhores em relação a versão original?